<a href="https://colab.research.google.com/github/Ruturaj2472/DeepEval_RAG/blob/main/DeepEval_RAG_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## 1. Setup & Environment

Installs:
- `deepeval` - the evaluation framework
- `langgraph`, `langchain`, `langchain-openai` - to rebuild the RAG pipeline
- `qdrant-client`, `fastembed`, `sentence-transformers` - retrieval + rerank stack

In [1]:
!pip install -q deepeval langgraph langchain langchain-openai langchain-core \
    qdrant-client fastembed sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.4/127.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 406.5/406.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.0/806.0 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.1/324.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not found - add it via the Colab Secrets (key icon) panel."
print("OpenAI API key loaded from Colab secrets ✔")

OpenAI API key loaded from Colab secrets ✔


In [3]:
from typing import TypedDict, List
from qdrant_client import QdrantClient, models as qmodels
from fastembed import TextEmbedding, SparseTextEmbedding
from sentence_transformers import CrossEncoder
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

# DeepEval is an evaluation framework for testing the quality
# of LLM applications and Retrieval-Augmented Generation (RAG) systems.
from deepeval import evaluate

# LLMTestCase defines the inputs, outputs, and retrieval context
from deepeval.test_case import LLMTestCase

from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    ContextualRelevancyMetric,
    HallucinationMetric,
    GEval,
)
# Defines which fields from an LLM test case should be evaluated
# when creating custom GEval metrics.
from deepeval.test_case import LLMTestCaseParams

/tmp/ipykernel_1412/3862917891.py:26: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams


##2. Building a Minimal RAG Pipeline

In [4]:
# Initialize the dense embedding model.
dense_model = TextEmbedding(model_name="BAAI/bge-small-en-v1.5")
DENSE_DIM = 384

# Initialize the sparse embedding model.
sparse_model= SparseTextEmbedding(model_name="Qdrant/bm25")

# EMBEDDING HELPER FUNCTIONS

def embed_dense(texts):
    return [vector.tolist() for vector in dense_model.embed(texts)]

def embed_sparse(texts):
    return list(sparse_model.embed(texts))

# LOAD CROSS-ENCODER RERANKER
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

# INITIALIZE THE LANGUAGE MODEL
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

# Confirm that all models have been initialized successfully.
print("Models loaded ✔")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Models loaded ✔


In [6]:
# Name of the Qdrant collection that will store
# documents and their corresponding embeddings.
COLLECTION_NAME = "langgraph_hybrid_eval_demo"
client = QdrantClient(":memory:")

# Create a collection capable of storing both
# dense and sparse embeddings for Hybrid Search.
client.create_collection(
    collection_name=COLLECTION_NAME,

    # Configuration for dense vector embeddings.
    vectors_config={
        "dense": qmodels.VectorParams(
            size=DENSE_DIM,                     # Dimension of dense embeddings
            distance=qmodels.Distance.COSINE    # Similarity metric for vector search
        )
    },

    # Configuration for sparse vector embeddings.
    sparse_vectors_config={
        "sparse": qmodels.SparseVectorParams()
    },
)

# Confirm successful collection creation.
print(f"Collection '{COLLECTION_NAME}' created ✔")

documents = [
    {"id": 0, "text": "LangGraph is a library for building stateful, multi-actor applications with LLMs, built on top of LangChain.", "topic": "langgraph"},
    {"id": 1, "text": "A StateGraph in LangGraph defines nodes, edges, and a shared state object that flows through the graph.", "topic": "langgraph"},
    {"id": 2, "text": "Unlike LCEL chains, which are directed acyclic graphs, LangGraph supports cycles and conditional branching.", "topic": "langgraph"},
    {"id": 3, "text": "Qdrant is an open-source vector database written in Rust, optimized for high-performance similarity search.", "topic": "qdrant"},
    {"id": 4, "text": "Qdrant supports named vectors, allowing multiple embeddings (e.g., dense and sparse) to be stored per point.", "topic": "qdrant"},
    {"id": 5, "text": "Hybrid search combines dense semantic vectors with sparse lexical vectors to improve retrieval accuracy.", "topic": "hybrid_search"},
    {"id": 6, "text": "Reciprocal Rank Fusion (RRF) is a common method for merging ranked lists from multiple retrieval systems.", "topic": "hybrid_search"},
    {"id": 7, "text": "BM25 is a classic sparse retrieval algorithm based on term frequency and inverse document frequency.", "topic": "hybrid_search"},
    {"id": 8, "text": "Dense embeddings encode semantic meaning, allowing retrieval of conceptually similar text even without shared keywords.", "topic": "embeddings"},
    {"id": 9, "text": "Cross-encoder rerankers score a query-document pair jointly, producing more accurate relevance scores than bi-encoders.", "topic": "reranking"},
    {"id": 10, "text": "BAAI/bge-reranker-base is an open-source cross-encoder model commonly used for reranking retrieved passages.", "topic": "reranking"},
    {"id": 11, "text": "Reranking is typically applied to a small candidate set (e.g., top 20-50) after initial retrieval, since cross-encoders are more compute-intensive.", "topic": "reranking"},
    {"id": 12, "text": "Agentic RAG systems combine retrieval-augmented generation with agent-style decision making, such as deciding whether to retrieve at all.", "topic": "agents"},
    {"id": 13, "text": "LangGraph's checkpointing feature allows persisting graph state, enabling resumability and human-in-the-loop workflows.", "topic": "langgraph"},
    {"id": 14, "text": "Conditional edges in LangGraph route execution to different nodes based on the current state, enabling dynamic control flow.", "topic": "langgraph"},
    {"id": 15, "text": "Open-source embedding models like bge-small-en-v1.5 can run efficiently on CPU, making them practical for local development.", "topic": "embeddings"},
    {"id": 16, "text": "Vector databases like Qdrant, Weaviate, and Milvus are optimized for approximate nearest neighbor search at scale.", "topic": "qdrant"},
    {"id": 17, "text": "SPLADE is a sparse retrieval model that learns term expansions, unlike BM25 which relies purely on exact term matches.", "topic": "hybrid_search"},
]

# Extract the text content from each document.
# These texts will be converted into dense and sparse embeddings.
texts = [doc["text"] for doc in documents]

# Generate dense embeddings
dense_vectors = embed_dense(texts)

# Generate sparse embeddings
sparse_vectors = embed_sparse(texts)

# Create Qdrant Point objects.
points = [
    qmodels.PointStruct(
        id=doc["id"],

        # Store both dense and sparse vectors
        # for Hybrid Search.
        vector={
            "dense": dense_vector,

            "sparse": qmodels.SparseVector(
                indices=sparse_vector.indices.tolist(),
                values=sparse_vector.values.tolist(),
            ),
        },

        # Additional metadata stored alongside
        # each document.
        payload={
            "text": doc["text"],
            "topic": doc["topic"],
        },
    )

    for doc, dense_vector, sparse_vector in zip(
        documents,
        dense_vectors,
        sparse_vectors,
    )
]

# Insert all documents into the Qdrant collection.
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

# Display the number of successfully stored documents.
print(f"Upserted {len(points)} points.")

Collection 'langgraph_hybrid_eval_demo' created ✔
Upserted 18 points.


In [7]:
def hybrid_search(
    query: str,
    top_k: int = 10,
    prefetch_k: int = 20,
):

    # Generate dense and sparse query embeddings.
    q_vec = embed_dense([query])[0]
    q_sparse = embed_sparse([query])[0]

    # Execute Hybrid Search.
    results = client.query_points(
        collection_name=COLLECTION_NAME,

        prefetch=[
            # Dense semantic retrieval
            qmodels.Prefetch(
                query=q_vec,
                using="dense",
                limit=prefetch_k,
            ),

            # Sparse keyword retrieval
            qmodels.Prefetch(
                query=qmodels.SparseVector(
                    indices=q_sparse.indices.tolist(),
                    values=q_sparse.values.tolist(),
                ),
                using="sparse",
                limit=prefetch_k,
            ),
        ],

        # Merge both rankings using RRF.
        query=qmodels.FusionQuery(
            fusion=qmodels.Fusion.RRF
        ),

        # Return the final top-k results.
        limit=top_k,
    )

    return results.points

# DEFINE LANGGRAPH STATE

class RAGState(TypedDict):
    query: str
    retrieved_docs: List[dict]
    reranked_docs: List[dict]
    answer: str

# NODE 1: RETRIEVE DOCUMENTS

def retrieve_node(state: RAGState) -> dict:

    # Retrieve candidate documents using Hybrid Search.
    results = hybrid_search(
        state["query"],
        top_k=10,
    )

    # Extract only the information needed for downstream nodes.
    docs = [
        {
        "text": r.payload["text"],
        "topic": r.payload["topic"],
        "score": r.score,
        }
        for r in results
    ]

    return {
        "retrieved_docs": docs,
    }

# NODE 2: RERANK DOCUMENTS

def rerank_node(state: RAGState) -> dict:

    # Create query-document pairs.
    pairs = [
        (state["query"], d["text"])
        for d in state["retrieved_docs"]
    ]

    # Predict semantic relevance scores.
    scores = reranker.predict(pairs)

    # Sort documents by descending relevance.
    combined = sorted(
        zip(state["retrieved_docs"], scores),
        key=lambda item: item[1],
        reverse=True,
    )

    # Keep only the top 3 documents.
    top_docs = [
        {
            **doc,
            "rerank_score": float(score),
        }
        for doc,score in combined[:3]
    ]

    return {
        "reranked_docs": top_docs
    }

# NODE 3: GENERATE ANSWER

def generate_node(state: RAGState) -> dict:

    # Combine retrieved documents into a single context block.
    context = "\n".join(
        f"- {doc['text']}"
        for doc in state["reranked_docs"]
    )

    # Construct the prompt.
    prompt = (
        "Answer the question using only the context below. "
        "If the context is insufficient, say so.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {state['query']}\n\n"
        "Answer:"
    )

    # Generate the answer.
    response = llm.invoke(prompt)

    return {
        "answer": response.content
    }

# BUILD THE LANGGRAPH WORKFLOW

# Create a graph using the shared workflow state.
graph_builder = StateGraph(RAGState)

# Register workflow nodes.
graph_builder.add_node("retrieve", retrieve_node)
graph_builder.add_node("rerank", rerank_node)
graph_builder.add_node("generate", generate_node)

# Define execution order.
graph_builder.add_edge(START, "retrieve")
graph_builder.add_edge("retrieve", "rerank")
graph_builder.add_edge("rerank", "generate")
graph_builder.add_edge("generate", END)

# Compile the graph into an executable workflow.
graph = graph_builder.compile()

print("RAG pipeline rebuilt and compiled ✔")

RAG pipeline rebuilt and compiled ✔


---
## 3. Building an Evaluation Dataset
DeepEval's RAG metrics need, per example:
- `input` - the query
- `actual_output` - what the pipeline generated
- `retrieval_context` - the chunks the pipeline actually retrieved (used by faithfulness/relevancy metrics)
- `expected_output` - a reference/"gold" answer (needed for `ContextualRecallMetric` and useful for `GEval`)

We define a small hand-curated eval set: queries paired with reference answers. We'll run the pipeline live to get `actual_output` and `retrieval_context` for each.

In [8]:
eval_dataset = [
    {
        "query": "What is the difference between LangGraph and a LangChain LCEL chain?",
        "expected_output": "LCEL chains are linear, acyclic pipelines, while LangGraph supports stateful graphs with cycles and conditional branching, making it suited for agentic workflows.",
    },
    {
        "query": "How does hybrid search combine dense and sparse retrieval?",
        "expected_output": "Hybrid search runs dense (semantic) and sparse (lexical/BM25-style) retrieval separately, then merges the two ranked lists using a fusion method such as Reciprocal Rank Fusion (RRF).",
    },
    {
        "query": "What open-source model can be used for reranking retrieved documents?",
        "expected_output": "Open-source cross-encoder models like cross-encoder/ms-marco-MiniLM-L-6-v2 or BAAI/bge-reranker-base can be used to rerank retrieved documents.",
    },
    {
        "query": "Why would I use RRF fusion instead of dense search alone?",
        "expected_output": "RRF fusion combines dense and sparse retrieval results, capturing both semantic similarity and exact keyword matches, which can outperform dense-only search especially for queries with specific terms.",
    },
    {
        "query": "What is Qdrant and what makes it different from a regular database?",
        "expected_output": "Qdrant is an open-source vector database optimized for high-performance similarity search over embeddings, unlike regular databases which are optimized for exact-match or relational queries.",
    },
    {
        "query": "What programming language is Qdrant written in?",
        "expected_output": "Qdrant is written in Rust.",
    },
]

print(f"Eval dataset size: {len(eval_dataset)} examples")

Eval dataset size: 6 examples


### Running the Pipeline to Collect Outputs

For each query we run the compiled LangGraph pipeline and capture the reranked context plus the generated answer - these become the `retrieval_context` and `actual_output` for each `LLMTestCase`.

In [9]:
# Store the outputs generated for each evaluation example.
results = []

# Execute the RAG pipeline for every evaluation example.
for example in eval_dataset:

    # Run the complete LangGraph workflow.
    output = graph.invoke(
        {
            "query": example["query"]
        }
    )

    # Save the information required for DeepEval metrics.
    results.append(
        {
            "query": example["query"],

            # Ground-truth/reference answer.
            "expected_output": example["expected_output"],

            # Answer generated by the RAG pipeline.
            "actual_output": output["answer"],

            # Context actually used by the LLM after reranking.
            "retrieval_context": [
                document["text"]
                for document in output["reranked_docs"]
            ],
        }
    )

# REVIEW GENERATED RESULTS

# Display the generated answer and retrieval context
# for each evaluation example before running DeepEval.
for result in results:

    print(f"\nQ: {result['query']}")
    print(f"A: {result['actual_output']}")

    # Display the number of retrieved context chunks
    # supplied to the language model.
    print(
        f"Context used: {len(result['retrieval_context'])} chunks"
    )


Q: What is the difference between LangGraph and a LangChain LCEL chain?
A: The difference between LangGraph and a LangChain LCEL chain is that LangGraph supports cycles and conditional branching, while LCEL chains are directed acyclic graphs. Additionally, LangGraph is designed for building stateful, multi-actor applications with LLMs and includes a StateGraph that defines nodes, edges, and a shared state object.
Context used: 3 chunks

Q: How does hybrid search combine dense and sparse retrieval?
A: Hybrid search combines dense semantic vectors, which capture the meaning of the content, with sparse lexical vectors, which focus on specific terms and their occurrences, to improve retrieval accuracy.
Context used: 3 chunks

Q: What open-source model can be used for reranking retrieved documents?
A: BAAI/bge-reranker-base is an open-source model that can be used for reranking retrieved documents.
Context used: 3 chunks

Q: Why would I use RRF fusion instead of dense search alone?
A: You 

---
## 4. Constructing DeepEval Test Cases

In [11]:
#Convert each RAG pipeline result into a DeepEval LLMTestCase.
#
# Every test case contains:
# - User input (query)
# - Model-generated answer
# - Expected (reference) answer
# - Retrieval context supplied to the LLM
#
# These test cases will be used by DeepEval metrics to evaluate
# both retrieval quality and answer quality.

test_cases = [
    LLMTestCase(
        input=result["query"],
        actual_output=result["actual_output"],
        expected_output=result["expected_output"],
        retrieval_context=result["retrieval_context"],
    )
    for result in results
]

# VERIFY THE GENERATED TEST CASES

# Display the total number of evaluation test cases created.
print(f"Built {len(test_cases)} LLMTestCase objects.")

# Inspect the contents of the first test case
# to ensure everything has been populated correctly.
print("\nExample test case fields:")

print("input:")
print(test_cases[0].input)

print("\nactual_output:")
print(test_cases[0].actual_output[:120], "...")

print("\nretrieval_context:")
print(test_cases[0].retrieval_context)

Built 6 LLMTestCase objects.

Example test case fields:
input:
What is the difference between LangGraph and a LangChain LCEL chain?

actual_output:
The difference between LangGraph and a LangChain LCEL chain is that LangGraph supports cycles and conditional branching, ...

retrieval_context:
['Unlike LCEL chains, which are directed acyclic graphs, LangGraph supports cycles and conditional branching.', 'LangGraph is a library for building stateful, multi-actor applications with LLMs, built on top of LangChain.', 'A StateGraph in LangGraph defines nodes, edges, and a shared state object that flows through the graph.']


---
## 5. Core RAG Metrics

All metrics below use an LLM internally (defaulting to `gpt-4o-mini` unless configured otherwise) to judge the relationship between `input`, `actual_output`, `retrieval_context`, and `expected_output`. Each metric produces a `0-1` score, a pass/fail against `threshold`, and a natural-language `reason`.


In [12]:
# Faithfulness Metric
faithfulness = FaithfulnessMetric(
    threshold=0.7
)

# Answer Relevancy Metric
answer_relevancy = AnswerRelevancyMetric(
    threshold=0.7
)

# Contextual Precision Metric
contextual_precision = ContextualPrecisionMetric(
    threshold=0.7
)

# Contextual Recall Metric
contextual_recall = ContextualRecallMetric(
    threshold=0.7
)

# Contextual Relevancy Metric
contextual_relevancy = ContextualRelevancyMetric(
    threshold=0.7
)

# GROUP ALL RAG METRICS
rag_metrics = [
    faithfulness,
    answer_relevancy,
    contextual_precision,
    contextual_recall,
    contextual_relevancy,
]

# Display the initialized metric names.
print(
    "RAG metrics initialized:",
    [metric.__class__.__name__ for metric in rag_metrics]
)

RAG metrics initialized: ['FaithfulnessMetric', 'AnswerRelevancyMetric', 'ContextualPrecisionMetric', 'ContextualRecallMetric', 'ContextualRelevancyMetric']


In [13]:
# Run evaluation across all test cases and all RAG metrics
eval_results = evaluate(test_cases=test_cases, metrics=rag_metrics)

✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Relevancy Metric! (using gpt-5.4, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 5 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              How does hybrid search combine dense and sparse retrieval?                           │
│  │     Actual Output:      Hybrid search combines dense semantic vectors, which capture the meaning of the      │
│  │                         content, with sparse lexical vectors, which focus on specific terms and their        │
│  │                         occurrences, to improve retrieval accuracy.                                          │
│  │     Expected Output:    Hybrid search runs dense (semantic) and sparse (lexical/BM25-style) retrieval        │
│  │                         separately, then merges the two ranked lists using a fusion method such as           │
│  │                         Reciprocal Rank Fusion (RRF).                                                        │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric               ┃ Score ┃ Threshold ┃ Reason                                                │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        PASS  │ Faithfulness         │ 1.00  │ 0.70      │ The score is 1.00 because there are no contradi...    │
│        PASS  │ Answer Relevancy     │ 1.00  │ 0.70      │ The score is 1.00 because the response directly...    │
│        PASS  │ Contextual Precision │ 1.00  │ 0.70      │ The score is 1.00 because the most relevant nod...    │
│        FAIL  │ Contextual Recall    │ 0.00  │ 0.70      │ The score is 0.00 because sentence 1 is not           │
│              │                      │       │           │ supported by the nodes in retrieval context: node 1   │
│              │                      │       │           │ only says hybrid search combines dense semantic       │
│              │                      │       │           │ vectors with sparse lexical vectors, and node 2       │
│              │                      │       │           │ only identifies BM25 as a sparse retrieval            │
│              │                      │       │           │ algorithm, but neither node states that dense and     │
│              │                      │       │           │ sparse retrieval are run separately, that two         │
│              │                      │       │           │ ranked lists are merged, or that fusion methods       │
│              │                      │       │           │ such as RRF are used.                                 │
│        PASS  │ Contextual Relevancy │ 1.00  │ 0.70      │ The score is 1.00 because the context directly ...    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭──────────────────────────────────────────────────────────

⚠ WARNING: No hyperparameters logged.
» ]8;id=838476;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 9.53s | token cost: 0.20833500000000005 USD)
» Test Results (6 total tests):
   » Pass Rate: 33.33% | Passed: 2 | Failed: 4

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

### Inspecting a Single Metric in Detail

Running a metric directly (rather than through `evaluate()`) gives access to `.score` and `.reason` for a single test case - useful when debugging why a specific example scored low.


In [14]:
sample_case = test_cases[1]

# FAITHFULNESS EVALUATION
faithfulness.measure(sample_case)

print("Faithfulness score:", faithfulness.score)
print("Reason:", faithfulness.reason)

# CONTEXTUAL RECALL EVALUATION
contextual_recall.measure(sample_case)

print("\nContextual Recall score:", contextual_recall.score)
print("Reason:", contextual_recall.reason)

Output()

Output()

Faithfulness score: 1.0
Reason: The score is 1.00 because there are no contradictions, so the actual output appears fully consistent with the retrieval context—great job keeping it accurate and faithful.



Contextual Recall score: 0.0
Reason: The score is 0.00 because sentence 1 is not attributable to the nodes in retrieval context: node 1 only indicates that hybrid search combines dense semantic and sparse lexical signals, and node 2 links BM25 to sparse retrieval, but no node in retrieval context states that the dense and sparse retrieval are executed separately, that they produce two ranked lists, or that those lists are merged with a fusion method such as RRF.


---
## 6. LLM-as-a-Judge with `GEval`

The built-in metrics above cover faithfulness/relevancy/recall/precision well, but sometimes you need to score something more specific to your use case - tone, instructional clarity, conciseness, whether an answer correctly cites which topic it came from, etc. `GEval` lets you define **custom evaluation criteria in plain English**; DeepEval turns that into an LLM-judged rubric (using chain-of-thought scoring internally) without you having to write a custom metric class.


In [15]:
# DEFINE CUSTOM LLM-AS-A-JUDGE METRICS

# CORRECTNESS METRIC
correctness_judge = GEval(
    name="Correctness",

    criteria=(
        "Determine whether the actual output is factually consistent "
        "with the expected output. The actual output does not need "
        "to match wording, but should not contradict or omit key "
        "facts present in the expected output."
    ),

    # Fields from the test case that will be supplied
    # to the evaluation model.
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],

    # Minimum passing score.
    threshold=0.7,
)

# INSTRUCTIONAL CLARITY METRIC
clarity_judge = GEval(
    name="Instructional Clarity",

    criteria=(
        "Evaluate whether the actual output is clear, concise, and "
        "appropriately explains the answer for someone learning the "
        "topic, without unnecessary hedging, filler, or repetition."
    ),

    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],

    threshold=0.7,
)

# GROUP CUSTOM EVALUATION METRICS

# Store all custom LLM-as-a-Judge metrics in a list.
judge_metrics = [
    correctness_judge,
    clarity_judge,
]

# Display the initialized metric names.
print(
    "LLM-as-judge metrics initialized:",
    [metric.name for metric in judge_metrics],
)


LLM-as-judge metrics initialized: ['Correctness', 'Instructional Clarity']


In [17]:
judge_results = evaluate(test_cases=test_cases, metrics=judge_metrics)

✨ You're running DeepEval's latest Correctness [GEval] Metric! (using gpt-5.4, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Instructional Clarity [GEval] Metric! (using gpt-5.4, strict=False, 
async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 2 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_1                                                                                                 │
│  ├──   Input:              How does hybrid search combine dense and sparse retrieval?                           │
│  │     Actual Output:      Hybrid search combines dense semantic vectors, which capture the meaning of the      │
│  │                         content, with sparse lexical vectors, which focus on specific terms and their        │
│  │                         occurrences, to improve retrieval accuracy.                                          │
│  │     Expected Output:    Hybrid search runs dense (semantic) and sparse (lexical/BM25-style) retrieval        │
│  │                         separately, then merges the two ranked lists using a fusion method such as           │
│  │                         Reciprocal Rank Fusion (RRF).                                                        │
│  └── Metrics                                                                                                    │
│       Status ┃ Metric                        ┃ Score ┃ Threshold ┃ Reason                                       │
│      ━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  │
│        FAIL  │ Correctness [GEval]           │ 0.56  │ 0.70      │ The response captures the high-level idea    │
│              │                               │       │           │ from the input that hybrid search combines   │
│              │                               │       │           │ dense semantic retrieval with sparse         │
│              │                               │       │           │ lexical retrieval, correctly                 │
│              │                               │       │           │ distinguishing meaning-based versus          │
│              │                               │       │           │ term-based signals. However, it omits the    │
│              │                               │       │           │ key expected mechanism that dense and        │
│              │                               │       │           │ sparse retrieval are run separately and      │
│              │                               │       │           │ then their ranked results are merged using   │
│              │                               │       │           │ a fusion method such as Reciprocal Rank      │
│              │                               │       │           │ Fusion (RRF). Because the expected output    │
│              │                               │       │           │ specifically focuses on separate retrieval   │
│              │                               │       │           │ plus rank-list fusion, the answer is only    │
│              │                               │       │           │ partially aligned.                           │
│        PASS  │ Instructional Clarity [GEval] │ 1.00  │ 0.70      │ The response directly answers how hybrid     │
│              │                               │       │   

⚠ WARNING: No hyperparameters logged.
» ]8;id=956274;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.88s | token cost: 0.027875 USD)
» Test Results (6 total tests):
   » Pass Rate: 83.33% | Passed: 5 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

---
## 7. Aggregating Results into a Scorecard

In [18]:
import pandas as pd

# Combine the built-in RAG metrics with the custom
# LLM-as-a-Judge metrics into a single evaluation suite.
all_metrics = rag_metrics + judge_metrics

# Store the evaluation scores for each test case.
rows = []

# Evaluate every test case across all metrics.
for test_case in test_cases:

    # Create a row for the final scorecard.
    row = {
        "query": test_case.input
    }

    # Compute each evaluation metric.
    for metric in all_metrics:

        # Run the metric on the current test case.
        metric.measure(test_case)

        # Use the metric's custom name if available;
        # otherwise use the class name.
        metric_name = getattr(
            metric,
            "name",
            metric.__class__.__name__,
        )

        # Store the score (rounded for readability).
        row[metric_name] = (
            round(metric.score, 3)
            if metric.score is not None
            else None
        )

    # Save the completed evaluation row.
    rows.append(row)

# BUILD THE EVALUATION SCORECARD

# Convert all evaluation results into a Pandas DataFrame.
scorecard = pd.DataFrame(rows)

# Display the complete evaluation scorecard.
scorecard

Output()

Output()

Output()

Output()

Output()

...

Output()

Output()

Output()

Output()

,query,FaithfulnessMetric,AnswerRelevancyMetric,ContextualPrecisionMetric,ContextualRecallMetric,ContextualRelevancyMetric,Correctness,Instructional Clarity
0,What is the difference between LangGraph and a...,1.0,1.0,1.000,1.0,1.000,1.000,0.938
1,How does hybrid search combine dense and spars...,1.0,1.0,1.000,0.0,1.000,0.563,0.996
2,What open-source model can be used for reranki...,1.0,1.0,1.000,1.0,0.333,0.844,1.000
3,Why would I use RRF fusion instead of dense se...,1.0,1.0,0.583,0.0,1.000,0.892,0.900
4,What is Qdrant and what makes it different fro...,1.0,1.0,0.833,1.0,1.000,0.995,0.902
5,What programming language is Qdrant written in?,1.0,1.0,1.000,1.0,0.333,1.000,1.000


In [19]:
# Average score per metric - quick view of systematic strengths/weaknesses
scorecard.drop(columns=["query"]).mean().sort_values().to_frame(name="avg_score")

,avg_score
ContextualRecallMetric,0.666667
ContextualRelevancyMetric,0.777667
Correctness,0.882333
ContextualPrecisionMetric,0.902667
Instructional Clarity,0.956000
AnswerRelevancyMetric,1.000000
FaithfulnessMetric,1.000000
